<a href="https://colab.research.google.com/github/SnehaDharne/OcularDiseaseRecognition/blob/main/hyp_resnet50_mirnet_msr.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Fast extract directly to Colab local SSD (takes ~5-10 seconds)
!unzip -q "/content/drive/MyDrive/archive (2).zip" -d "/content/data"
print("Extraction complete!")


Mounted at /content/drive
Extraction complete!


In [ ]:
import os
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.resnet50 import ResNet50
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# Path to the extracted datasets folder
data_dir = "/content/data/datasets"

# Data Augmentation & Normalization
datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    horizontal_flip=True,
    validation_split=0.2  # 80% train, 20% validation
)

train_generator = datagen.flow_from_directory(
    data_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

val_generator = datagen.flow_from_directory(
    data_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

print("Class mapping:", train_generator.class_indices)


Found 5118 images belonging to 8 classes.
Found 1274 images belonging to 8 classes.
Class mapping: {'ageDegeneration': 0, 'cataract': 1, 'diabetes': 2, 'glaucoma': 3, 'hypertension': 4, 'myopia': 5, 'normal': 6, 'others': 7}


In [ ]:
import numpy as np
from sklearn.utils import class_weight

# Calculate balanced weights inversely proportional to class frequencies
class_weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_generator.classes),
    y=train_generator.classes
)
class_weight_dict = dict(enumerate(class_weights))

print("Calculated Class Weights:")
for idx, weight in class_weight_dict.items():
    class_name = list(train_generator.class_indices.keys())[idx]
    print(f" - {class_name:<20} : {weight:.2f}x penalty")


Calculated Class Weights:
 - ageDegeneration      : 3.00x penalty
 - cataract             : 2.72x penalty
 - diabetes             : 0.50x penalty
 - glaucoma             : 2.81x penalty
 - hypertension         : 6.21x penalty
 - myopia               : 3.44x penalty
 - normal               : 0.28x penalty
 - others               : 1.13x penalty


In [ ]:
from tensorflow.keras.applications.resnet50 import ResNet50
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import AUC
import tensorflow as tf

# Base ResNet50
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

# GlobalAveragePooling2D aggregates spatial features cleanly (no 51M parameter explosion!)
model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    BatchNormalization(),
    Dense(256, activation='relu'),
    Dropout(0.3),
    Dense(128, activation='relu'),
    Dense(8, activation='softmax')
])

model.compile(
    optimizer=Adam(learning_rate=0.0003),
    loss='categorical_crossentropy',
    metrics=['accuracy', AUC(name='auc')]
)

model.summary()


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 2048)           │         8,192 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 256)            │       524,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 8)              │         1,032 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 24,154,376 (92.14 MB)

 Trainable params: 562,568 (2.15 MB)

 Non-trainable params: 23,591,808 (90.00 MB)

In [ ]:
import numpy as np
from sklearn.utils import class_weight
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# 1. Compute square-root smoothed weights (prevents gradient explosions on rare classes)
raw_weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_generator.classes),
    y=train_generator.classes
)
# Square-root dampening gives balanced penalty without destabilizing the network
smoothed_weights = {k: float(np.sqrt(v)) for k, v in enumerate(raw_weights)}

print("Smoothed weights:", smoothed_weights)

# 2. Checkpoint & Train
save_path = "/content/drive/MyDrive/best_resnet50_balanced.keras"

checkpoint = ModelCheckpoint(
    filepath=save_path,
    monitor='val_auc',
    mode='max',
    save_best_only=True,
    verbose=1
)

early_stop = EarlyStopping(
    monitor='val_auc',
    mode='max',
    patience=4,
    restore_best_weights=True
)

history = model.fit(
    train_generator,
    epochs=10,
    validation_data=val_generator,
    class_weight=smoothed_weights,
    callbacks=[checkpoint, early_stop]
)


Smoothed weights: {0: 1.7330669712277607, 1: 1.6499516433919859, 2: 0.7050434363117262, 3: 1.675088371588669, 4: 2.492220906850693, 5: 1.8545932516409787, 6: 0.527516030319889, 7: 1.062217905284447}
Epoch 1/10
160/160 ━━━━━━━━━━━━━━━━━━━━ 0s 491ms/step - accuracy: 0.3636 - auc: 0.7465 - loss: 1.7042
Epoch 1: val_auc improved from None to 0.79646, saving model to /content/drive/MyDrive/best_resnet50_balanced.keras

Epoch 1: finished saving model to /content/drive/MyDrive/best_resnet50_balanced.keras
160/160 ━━━━━━━━━━━━━━━━━━━━ 118s 662ms/step - accuracy: 0.4045 - auc: 0.7877 - loss: 1.6875 - val_accuracy: 0.4435 - val_auc: 0.7965 - val_loss: 1.6880
Epoch 2/10
160/160 ━━━━━━━━━━━━━━━━━━━━ 0s 453ms/step - accuracy: 0.4238 - auc: 0.8142 - loss: 1.6220
Epoch 2: val_auc did not improve from 0.79646
160/160 ━━━━━━━━━━━━━━━━━━━━ 90s 564ms/step - accuracy: 0.4222 - auc: 0.8187 - loss: 1.5970 - val_accuracy: 0.2755 - val_auc: 0.7902 - val_loss: 1.7804
Epoch 3/10
160/160 ━━━━━━━━━━━━━━━━━━━━ 0s 

In [ ]:
from google.colab import files
files.download("/content/drive/MyDrive/best_resnet50_balanced.keras")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# 1. Verify loading the saved model
loaded_model = tf.keras.models.load_model("/content/drive/MyDrive/best_resnet50_model.keras")
print("Model loaded and verified successfully!")

# 2. (Optional) Download directly to your local computer
from google.colab import files
files.download("/content/drive/MyDrive/best_resnet50_model.keras")


Model loaded and verified successfully!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>